# Campus Guardian: AI-Assisted Safety Classification

**Project type:** Machine learning prototype  
**Algorithm:** Multiclass Logistic Regression  
**Application area:** School and campus incident awareness

## Overview

This notebook explores whether a small set of environmental readings can help sort a campus situation into one of three response levels:

| Code | Predicted level | Meaning |
|---:|---|---|
| 0 | Safe | No immediate warning is indicated |
| 1 | Check | The readings deserve attention or inspection |
| 2 | Urgent | The readings indicate a potentially serious event |

The notebook follows a practical workflow: inspect the records, prepare the input variables, train a classifier, check its performance, and demonstrate how a prediction could be turned into a logged response.

**Important limitation:** This is an educational prototype using a dataset. It is not a certified safety device and must never replace alarms, trained staff, or emergency procedures.


## 1. Define the task

A school may collect readings such as temperature, smoke intensity, gas concentration, sound, crowd size, and motion. Looking at every reading manually can be slow, especially when several values change together.

The goal here is to train a model that estimates a safety category from those readings. The output is a decision-support label, not proof that an incident is happening. Any real deployment would need sensor validation, careful testing, privacy safeguards, and a human response process.


## 2. Load the tools we need

Pandas handles the records, Matplotlib creates simple charts, and scikit-learn provides the training and evaluation utilities.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

pd.set_option("display.max_columns", None)


## 3. Read the dataset

Place `SchoolSafetyEmergencyDataset.csv` in the notebook's working folder before running this cell. In Google Colab, upload the file or update the path below to match its location.


In [ ]:
data_path = "SchoolSafetyEmergencyDataset.csv"
records = pd.read_csv(data_path)

print(f"Rows: {records.shape[0]} | Columns: {records.shape[1]}")
display(records.head())


## 4. Check the data before modelling

First, confirm the column names and look at how many examples belong to each class. Class balance matters because a model can appear accurate if it mostly predicts the most common category.


In [ ]:
print("Columns in the file:")
print(records.columns.tolist())

print("\nMissing values by column:")
display(records.isna().sum().to_frame("missing_count"))

print("\nExamples in each target class:")
display(records["severity_label"].value_counts().to_frame("records"))


In [ ]:
class_counts = records["severity_label"].value_counts()

plt.figure(figsize=(7, 4))
class_counts.plot(kind="bar")
plt.title("Records by Safety Category")
plt.xlabel("Safety category")
plt.ylabel("Number of records")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 5. Select the model inputs

The classifier uses the numerical sensor fields below. `incident_type` and `severity_label` are descriptive fields; they are not included as predictors. The target column is `severity`.

If your CSV uses different column names, update the list to match the file rather than silently substituting other fields.


In [ ]:
sensor_fields = [
    "temperature_c",
    "smoke_level",
    "co_ppm",
    "gas_level",
    "sound_db",
    "crowd_count",
    "motion_index",
    "flame_detected",
    "event_duration_min"
]

target_field = "severity"

required_fields = sensor_fields + [target_field]
missing_fields = [name for name in required_fields if name not in records.columns]

if missing_fields:
    raise ValueError(f"Required columns not found: {missing_fields}")

model_data = records[required_fields].dropna().copy()
features = model_data[sensor_fields]
target = model_data[target_field]

print("Rows available for training:", len(model_data))
print("Input variables:", len(sensor_fields))


## 6. Create training and testing groups

The test group is kept separate so that evaluation uses records the model did not see during fitting. Stratification helps preserve the target-class proportions in both groups.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.20,
    random_state=17,
    stratify=target
)

print(f"Training examples: {X_train.shape[0]}")
print(f"Testing examples:  {X_test.shape[0]}")


## 7. Train the classifier

Sensor fields have different units and ranges, so the pipeline standardizes the inputs before fitting Logistic Regression. Keeping these steps in one pipeline ensures the same preprocessing is applied during both training and prediction.


In [ ]:
safety_model = Pipeline(steps=[
    ("standardize", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1200,
        class_weight="balanced",
        random_state=17
    ))
])

safety_model.fit(X_train, y_train)
test_predictions = safety_model.predict(X_test)


## 8. Review model performance

Accuracy gives an overall score, while precision, recall, and F1-score show how predictions behave for each class. For safety-related use cases, missed urgent cases can be especially important, so the full report should be reviewed rather than relying on accuracy alone.


In [ ]:
score = accuracy_score(y_test, test_predictions)
print(f"Test accuracy: {score:.3f}\n")

print(classification_report(
    y_test,
    test_predictions,
    labels=[0, 1, 2],
    target_names=["Safe", "Check", "Urgent"],
    zero_division=0
))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    labels=[0, 1, 2],
    display_labels=["Safe", "Check", "Urgent"],
    cmap="Blues",
    xticks_rotation=15
)
plt.title("Actual vs Predicted Safety Category")
plt.tight_layout()
plt.show()


## 9. Turn a model result into a response message

This function only creates text for the demonstration. It does not contact emergency services or send a notification.


In [ ]:
category_names = {
    0: "Safe",
    1: "Check",
    2: "Urgent"
}

def response_for(category_code):
    messages = {
        0: "STATUS: No active warning predicted.",
        1: "NOTICE: Review the area and verify the readings.",
        2: "URGENT: Follow the school's established emergency procedure."
    }
    return messages.get(int(category_code), "STATUS: Unknown category.")

for code in sorted(category_names):
    print(f"{category_names[code]} -> {response_for(code)}")


## 10. Test one hypothetical reading

Edit the example values to experiment with the model. These values are a demonstration only; they should not be interpreted as validated real-world thresholds.


In [ ]:
sample_reading = pd.DataFrame([{
    "temperature_c": 65,
    "smoke_level": 88,
    "co_ppm": 40,
    "gas_level": 92,
    "sound_db": 101,
    "crowd_count": 48,
    "motion_index": 90,
    "flame_detected": 1,
    "event_duration_min": 8
}])[sensor_fields]

chosen_class = int(safety_model.predict(sample_reading)[0])
class_probabilities = safety_model.predict_proba(sample_reading)[0]

print("Model category:", category_names.get(chosen_class, str(chosen_class)))
print(response_for(chosen_class))
print("\nEstimated class probabilities:")
for code, probability in zip(safety_model.named_steps["classifier"].classes_, class_probabilities):
    print(f"{category_names.get(int(code), str(code)):>8}: {probability:.1%}")


## 11. Simulate a sequence of readings

For this demonstration, a selection of dataset rows is treated as if they arrived one after another. Each prediction becomes an entry in a log. If a warning is active and a later prediction is safe, the log records that the earlier warning was cleared.

This is a simplified state example, not a substitute for a real alert policy with confirmation windows, staff acknowledgement, and fail-safe rules.


In [ ]:
# Arrange a mixed sequence so the log contains more than one category.
stream_parts = []
for code, sample_size, seed in [(1, 3, 11), (2, 4, 22), (0, 4, 33)]:
    class_rows = records[records[target_field] == code]
    if not class_rows.empty:
        stream_parts.append(class_rows.sample(
            n=min(sample_size, len(class_rows)),
            random_state=seed
        ))

stream_parts.append(records.sample(n=min(9, len(records)), random_state=44))
incoming_rows = pd.concat(stream_parts, ignore_index=True)

event_log = []
warning_open = False

for step_no, (_, row) in enumerate(incoming_rows.iterrows(), start=1):
    input_row = row[sensor_fields].to_frame().T
    predicted_code = int(safety_model.predict(input_row)[0])

    if predicted_code == 0:
        action = "Warning cleared" if warning_open else "No warning"
        warning_open = False
    elif predicted_code == 1:
        action = "Inspection requested"
        warning_open = True
    else:
        action = "Urgent response indicated"
        warning_open = True

    event_log.append({
        "step": step_no,
        "predicted_category": category_names.get(predicted_code, str(predicted_code)),
        "action": action,
        "temperature_c": row["temperature_c"],
        "smoke_level": row["smoke_level"],
        "flame_detected": row["flame_detected"],
        "source_incident": row.get("incident_type", "Not provided")
    })

log_table = pd.DataFrame(event_log)
display(log_table)


## 12. Latest-status panel

The panel below summarizes the last item in the simulated stream and displays the most recent log entries.


In [ ]:
if not log_table.empty:
    current = log_table.iloc[-1]

    print("-" * 52)
    print("CAMPUS GUARDIAN | CURRENT STATUS")
    print("-" * 52)
    print("Predicted category :", current["predicted_category"])
    print("System action      :", current["action"])
    print("Temperature        :", current["temperature_c"], "°C")
    print("Smoke reading      :", current["smoke_level"])
    print("Flame indicator    :", current["flame_detected"])
    print("-" * 52)

    print("\nLatest event records:")
    display(log_table.tail(8))
else:
    print("No readings were available for the demonstration.")


## 13. Findings and limitations

Use the results produced by your own run to complete this section.

- **Evaluation:** Record the accuracy and class-wise precision/recall from the report above.
- **Most important observation:** Explain which category is most often confused in the confusion matrix.
- **Data limitation:** Model performance depends on the quality and representativeness of the dataset. Simulated records cannot establish real-world safety performance.
- **Operational limitation:** The prototype produces local text output only. It does not connect to cameras, physical sensors, SMS, or emergency services.
- **Responsible use:** Any practical system needs independent validation, secure handling of data, clear escalation rules, and human oversight.

Do not report a result that you have not observed from the executed notebook.


## 14. Possible next steps

1. Validate the dataset labels and document how the records were created.
2. Compare Logistic Regression with another baseline classifier.
3. Review recall for the urgent category and investigate missed cases.
4. Add a timestamp and location identifier to the event log.
5. Build a small dashboard for staff to review and acknowledge warnings.
6. Test the system with safety professionals before considering any real deployment.

## Conclusion

This project demonstrates a complete introductory machine-learning workflow: data inspection, feature selection, train/test separation, model fitting, evaluation, single-record prediction, and simulated event logging. Its purpose is to show the workflow clearly—not to certify an emergency detection system.
